# Newsletter Pipeline Observability

Concrete examples pulled from the SQLite DB and (optionally) your vault.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebook lives in observability/
sys.path.insert(0, str(repo_root))


In [2]:
import os
import sqlite3
from textwrap import indent

try:
    import pandas as pd
except Exception:
    pd = None

def open_db(db_path: str) -> sqlite3.Connection:
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database not found: {db_path}")
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn

def rows(conn: sqlite3.Connection, query: str, params=()):
    return [dict(r) for r in conn.execute(query, params).fetchall()]

def first_row(conn: sqlite3.Connection, query: str, params=()):
    row = conn.execute(query, params).fetchone()
    return dict(row) if row else None

def file_preview(path: str, max_lines: int = 40) -> str:
    if not path or not os.path.exists(path):
        return "<missing>"
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    return "".join(lines[:max_lines]).rstrip()

def show_table(data, title=None):
    if title:
        print(title)
    if not data:
        print("(no rows)")
        return
    if pd is not None:
        display(pd.DataFrame(data))
    else:
        for row in data:
            print(row)

# Parameters
db_path = "../newsletter.db"
vault_path = "../vault"  # optional, e.g. r"C:\\Path\\To\\Vault"
max_samples = 3

conn = open_db(db_path)


In [3]:
# Gmail examples
gmail_count = first_row(conn, "SELECT COUNT(*) AS n FROM gmail_messages")
print(f"Total messages: {gmail_count['n'] if gmail_count else 0}")
gmail_samples = rows(
    conn,
    """
    SELECT message_id, internal_date, subject, from_email, issue_note_path
    FROM gmail_messages
    ORDER BY internal_date DESC
    LIMIT ?
    """,
    (max_samples,),
)
show_table(gmail_samples, title="Recent messages")

for msg in gmail_samples:
    if msg.get("issue_note_path"):
        preview = file_preview(msg["issue_note_path"])
        print("\nIssue note preview:")
        print(indent(preview or "<empty>", "  "))


Total messages: 20
Recent messages


,message_id,internal_date,subject,from_email,issue_note_path
0,19c11e9465c23bd6,1769827026000,Robbyant Open Sources LingBot World | DeepSee...,Marktechpost AI <marktechpost-newsletter@mail....,C:\Users\felze\source\repos\newsletter\vault\N...
1,19c0f4bb1e82a873,1769783144000,"Apple audio AI acquisition 🎧, OpenAI’s data ag...",TLDR AI <dan@tldrnewsletter.com>,C:\Users\felze\source\repos\newsletter\vault\N...
2,19c0eda69526339e,1769775720000,"Google’s AI SRE 🔍, Sentry’s CLI 🆕, 10 Years of...",TLDR DevOps <dan@tldrnewsletter.com>,C:\Users\felze\source\repos\newsletter\vault\N...



Issue note preview:
  ---
  type: newsletter-issue
  source: "Robbyant Open Sources LingBot World |  DeepSeek AI Releases DeepSeek-OCR 2"
  date: 2026-01-30
  gmail_message_id: "19c11e9465c23bd6"
  from: "Marktechpost AI <marktechpost-newsletter@mail.beehiiv.com>"
  tags: [newsletters]
  ---

  # Summary
  - Total links: 67
  - Domains: 10

  # Links
  ## ainews.sh

  - [https://ainews.sh/ProductDetail?id=697c618bd71aafc3264b92fa](https://ainews.sh/ProductDetail?id=697c618bd71aafc3264b92fa)
  - [https://ainews.sh/ProductDetail?id=697d21c5a07d776f43e470cd](https://ainews.sh/ProductDetail?id=697d21c5a07d776f43e470cd)
  - [https://ainews.sh/ProductDetail?id=697d249dcb670de635f290b1](https://ainews.sh/ProductDetail?id=697d249dcb670de635f290b1)
  - [https://ainews.sh/ProductDetail?id=697c6bd05dcfa5082abae5f2](https://ainews.sh/ProductDetail?id=697c6bd05dcfa5082abae5f2)
  - [https://ainews.sh/ProductDetail?id=697d22bdca2a6348f98bdb32](https://ainews.sh/ProductDetail?id=697d22bdca2a6348f98bd

In [4]:
# Link examples
link_count = first_row(conn, "SELECT COUNT(*) AS n FROM links")
print(f"Total links: {link_count['n'] if link_count else 0}")

status_counts = rows(
    conn,
    """
    SELECT COALESCE(fetch_status, 'unprocessed') AS status, COUNT(*) AS n
    FROM links
    GROUP BY status
    ORDER BY n DESC
    """,
)
show_table(status_counts, title="Status breakdown")

top_domains = rows(
    conn,
    """
    SELECT domain, COUNT(*) AS n
    FROM links
    GROUP BY domain
    ORDER BY n DESC
    LIMIT ?
    """,
    (max_samples,),
)
show_table(top_domains, title="Top domains")

link_samples = rows(
    conn,
    """
    SELECT url_canonical, domain, fetch_status, title, summary, note_path
    FROM links
    ORDER BY discovered_at DESC
    LIMIT ?
    """,
    (max_samples,),
)
show_table(link_samples, title="Recent links")

for link in link_samples:
    if link.get("summary"):
        print("\nSummary:")
        print(indent(link["summary"].strip(), "  "))
    if link.get("note_path"):
        preview = file_preview(link["note_path"])
        print("\nArticle note preview:")
        print(indent(preview or "<empty>", "  "))


Total links: 1037
Status breakdown


,status,n
0,unprocessed,1037


Top domains


,domain,n
0,tracking.tldrnewsletter.com,489
1,link.mail.beehiiv.com,175
2,github.com,34


Recent links


,url_canonical,domain,fetch_status,title,summary,note_path
0,https://tracking.tldrnewsletter.com/CL0/https:...,tracking.tldrnewsletter.com,None,None,None,None
1,https://tracking.tldrnewsletter.com/CL0/https:...,tracking.tldrnewsletter.com,None,None,None,None
2,https://tracking.tldrnewsletter.com/CL0/https:...,tracking.tldrnewsletter.com,None,None,None,None


## Web requests + parsing (pipeline demo)

The pipeline uses `fetch_article()` and `extract_main_text()` from `run.py`.
The cells below show how those methods behave on a single URL.
They do **live HTTP requests** when executed.

In [5]:
from run import fetch_article, extract_main_text

# Pick any URL you want to test
test_url = "https://example.com"

status, title, text = fetch_article(test_url)
print("status:", status)
print("title:", title)
print("text preview:", (text or "")[:500])


status: ok
title: Example Domain
text preview: This domain is for use in documentation examples without needing permission. Avoid use in operations.
Learn more


In [6]:
# Optional: if you already have HTML (e.g., saved response), parse it directly
html = "<html><head><title>Sample</title></head><body><h1>Hello</h1><p>World</p></body></html>"
title2, text2 = extract_main_text(html)
print("title:", title2)
print("text:", text2)


title: Sample
text: HelloWorld


In [7]:
# Optional vault checks
if vault_path:
    issues_root = os.path.join(vault_path, "Newsletters", "Issues")
    articles_root = os.path.join(vault_path, "Newsletters", "Articles")
    print(f"Issues root exists: {issues_root} -> {os.path.exists(issues_root)}")
    print(f"Articles root exists: {articles_root} -> {os.path.exists(articles_root)}")
else:
    print("Set vault_path to preview files under your vault.")


Issues root exists: ../vault\Newsletters\Issues -> True
Articles root exists: ../vault\Newsletters\Articles -> False


## JSONL log analysis

Aggregates over run logs produced by `--log-jsonl` on `ingest`, `process-links`, and `refresh`.
Answers "what broke this week?" using the fields added in Milestone B (`error_class`, `http_status`, `retry_count`, `llm_mode`, `fallback_used`, `model`, `prompt_version`, `llm_latency_ms`).

Same aggregations power `python -m observability.log_report`.


In [ ]:
# Load all JSONL logs written by the pipeline (--log-jsonl)
from observability.logs import filter_events, load_events

logs_glob = str(repo_root / "logs" / "*.jsonl")
events = load_events([logs_glob])
# Optional: trim to the last 30 days
events = filter_events(events, since_days=30)
url_events = [ev for ev in events if ev.get("event") == "url_processed"]
print(f"Loaded {len(events)} events ({len(url_events)} url_processed) from {logs_glob}")


In [ ]:
# Overall success rate + error class breakdown
from observability.log_stats import error_class_breakdown, success_rate

rate = success_rate(url_events)
print(
    f"total={rate['total']}  ok={rate['ok']}  fail={rate['fail']}  "
    f"success={rate['success_rate']:.1%}"
)
show_table(
    [
        {
            "command": cmd,
            **{k: v for k, v in bucket.items() if k != "success_rate"},
            "success_rate": f"{bucket['success_rate']:.1%}",
        }
        for cmd, bucket in sorted(rate["by_command"].items())
    ],
    title="Success rate by command",
)
show_table(error_class_breakdown(url_events), title="Error class breakdown")


In [ ]:
# Top failing domains
from observability.log_stats import top_failing_domains, trend_by_day

show_table(top_failing_domains(url_events, n=10), title="Top failing domains")


In [ ]:
# LLM mode + latency
from observability.log_stats import latency_summary, llm_mode_breakdown

llm = llm_mode_breakdown(url_events)
print(f"Successful summaries: {llm['total']}  fallback_used: {llm['fallback_used']}")
show_table(
    [{"mode": mode, "count": count} for mode, count in sorted(llm["by_mode"].items(), key=lambda kv: -kv[1])],
    title="LLM mode breakdown",
)
show_table(latency_summary(url_events), title="Latency by (model, prompt_version)")


In [ ]:
# Daily trend chart
import matplotlib.pyplot as plt

trend = trend_by_day(url_events)
if trend:
    dates = [row["date"] for row in trend]
    processed = [row["processed"] for row in trend]
    fails = [row["fail"] for row in trend]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(dates, processed, marker="o", label="processed")
    ax.plot(dates, fails, marker="o", label="fail")
    ax.set_xlabel("date (UTC)")
    ax.set_ylabel("count")
    ax.set_title("Daily pipeline volume and failures")
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("(no dated url_processed events)")
